# 第314章: Null-text Inversion と Prompt-to-Prompt (概念)

## 📋 この章で学ぶこと

- [ ] DDIM InversionがCFG（Classifier-Free Guidance）下で誤差を生む理由を説明できる
- [ ] Null-text Inversionの核心アイデア（無条件埋め込みの最適化）を理解できる
- [ ] Prompt-to-Promptの「Cross-Attention注入」による構造保持の直感を説明できる
- [ ] 簡易的な最適化ループを実装し、トラジェクトリー誤差を最小化できる

## 🎯 前提知識

- ✅ Notebook 310（DDIM Inversion）
- ✅ Notebook 312（テキスト補間、CFG）

⏱️ **推定学習時間**: 90分
📊 **難易度**: ★★★★★（専門的）
🎓 **カテゴリ**: 理論・発展

---

## 🌟 はじめに

312章では、Inversion後に条件（クラス）を変えてSamplingをしました。
しかし、**Classifier-Free Guidance (CFG: w>1) を使ってInversionを行うと、元の画像が正確に再構成できない**という大きな問題があります。

### 🚨 CFG環境下でのInversionのジレンマ

1. **w=1（CFGなし）でInversion**: 再構成は完璧だが、編集時にCFGを強く効かせられない
2. **w>1（CFGあり）でInversion**: 決定論的仮定が崩れ、累積誤差で再構成が失敗する

### 💡 解決策: Null-text Inversion

Google Research (2022) が提案した画期的な手法です：
> **「無条件ベクトル（Null-text = $ arnothing $）そのものを最適化して、Inversionの軌跡と一致させよう」**

### 📝 この章の構成

1. **問題の確認** — CFG下でのInversion失敗を可視化
2. **Null-text最適化の概念** — どうやって軌跡を合わせるか
3. **簡易実装** — MNISTモデルでの軌跡最適化
4. **Prompt-to-Promptの直感** — 構造を保ったまま編集する仕組み

In [ ]:
# ============================================================
# 環境設定・モデルの準備（312章と同一）
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import warnings

warnings.filterwarnings('ignore')

def setup_japanese_font():
    japanese_fonts = ['Hiragino Sans', 'Yu Gothic', 'MS Gothic', 'Noto Sans CJK JP', 'IPAexGothic']
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

def get_schedule(n_steps=100, beta_start=1e-4, beta_end=0.02):
    betas = np.linspace(beta_start, beta_end, n_steps)
    alphas = 1.0 - betas
    alpha_bars = np.cumprod(alphas)
    return {'betas': torch.tensor(betas, dtype=torch.float32),
            'alphas': torch.tensor(alphas, dtype=torch.float32),
            'alpha_bars': torch.tensor(alpha_bars, dtype=torch.float32)}

class ConditionalNoisePredictor(nn.Module):
    def __init__(self, dim=784, n_classes=10, te_dim=32, ce_dim=32):
        super().__init__()
        self.null_token = nn.Parameter(torch.zeros(ce_dim))
        self.class_emb = nn.Embedding(n_classes, ce_dim)
        self.time_mlp = nn.Sequential(nn.Linear(1, te_dim), nn.SiLU(), nn.Linear(te_dim, te_dim))
        self.net = nn.Sequential(
            nn.Linear(dim + te_dim + ce_dim, 512), nn.SiLU(),
            nn.Linear(512, 512), nn.SiLU(),
            nn.Linear(512, dim),
        )
    def forward(self, x, t, c=None, custom_null=None):
        t_emb = self.time_mlp(t.float().unsqueeze(-1) / 100.0)
        if custom_null is not None:
            c_emb = custom_null
        else:
            c_emb = self.null_token.unsqueeze(0).expand(x.shape[0], -1) if c is None else self.class_emb(c)
        return self.net(torch.cat([x, t_emb, c_emb], dim=-1))
    def forward_cfg(self, x, t, c, w=7.5, custom_null=None):
        eps_u = self.forward(x, t, c=None, custom_null=custom_null)
        eps_c = self.forward(x, t, c=c)
        return eps_u + w * (eps_c - eps_u)

# 学習（10 epoch）
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
schedule = get_schedule()
ab = schedule['alpha_bars'].to(device)
cond_model = ConditionalNoisePredictor().to(device)
optimizer = optim.Adam(cond_model.parameters(), lr=3e-4)

print("モデル学習中...")
for epoch in range(2):
    cond_model.train()
    for x, y in train_loader:
        x = x.view(-1, 784).to(device) * 2 - 1
        y = y.to(device)
        t = torch.randint(0, 100, (x.shape[0],)).to(device)
        noise = torch.randn_like(x)
        x_noisy = torch.sqrt(ab[t].unsqueeze(-1)) * x + torch.sqrt(1 - ab[t].unsqueeze(-1)) * noise
        mask_u = torch.rand(x.shape[0], device=device) < 0.15
        t_cmb = cond_model.time_mlp(t.float().unsqueeze(-1) / 100.0)
        c_cmb = torch.where(mask_u.unsqueeze(-1), cond_model.null_token.unsqueeze(0).expand(x.shape[0], -1), cond_model.class_emb(y))
        loss = nn.functional.mse_loss(cond_model.net(torch.cat([x_noisy, t_cmb, c_cmb], dim=-1)), noise)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
print("✅ 学習完了")
cond_model.eval()

---

## 1. CFG下でのInversion失敗（再構成の崩れ）

In [ ]:
# ============================================================
# DDIM Inversion と Sampling の関数
# ============================================================

@torch.no_grad()
def ddim_inversion_cfg(model, x_0, c_label, w=1.0, n_steps=20):
    """CFGスケールwでのInversion。Trajectoryを返す"""
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    c = torch.full((x_0.shape[0],), c_label, dtype=torch.long).to(device)
    x = x_0.clone()
    traj = [x.clone()]
    for i in range(n_steps):
        t_c, t_n = step_idx[i], step_idx[i + 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model.forward_cfg(x, t_t, c, w=w)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_n]) * x0_p + torch.sqrt(1 - ab[t_n]) * eps
        traj.append(x.clone())
    return x, traj

@torch.no_grad()
def ddim_sample_cfg(model, x_T, c_label, w=1.0, n_steps=20):
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)
    c = torch.full((x_T.shape[0],), c_label, dtype=torch.long).to(device)
    x = x_T.clone()
    for i in range(n_steps, 0, -1):
        t_c, t_p = step_idx[i], step_idx[i - 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)
        eps = model.forward_cfg(x, t_t, c, w=w)
        x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
        x = torch.sqrt(ab[t_p]) * x0_p + torch.sqrt(1 - ab[t_p]) * eps
    return x

# テスト画像
test_batch, test_labels = next(iter(test_loader))
test_imgs = test_batch.view(-1, 784).to(device) * 2 - 1
img_5 = test_imgs[(test_labels == 5).nonzero()[0].item():(test_labels == 5).nonzero()[0].item()+1]

w_values = [1.0, 3.0, 7.5, 12.0]
fig, axes = plt.subplots(2, len(w_values), figsize=(14, 5))

for col, w in enumerate(w_values):
    x_T, _ = ddim_inversion_cfg(cond_model, img_5, 5, w=w)
    x_recon = ddim_sample_cfg(cond_model, x_T, 5, w=w)

    axes[0, col].imshow(((img_5[0].cpu().numpy().reshape(28,28)+1)/2).clip(0,1), cmap='gray')
    axes[1, col].imshow(((x_recon[0].cpu().numpy().reshape(28,28)+1)/2).clip(0,1), cmap='gray')

    mse = nn.functional.mse_loss(x_recon, img_5).item()
    axes[0, col].set_title(f'w={w}', fontsize=12)
    axes[1, col].set_title(f'MSE: {mse:.4f}', fontsize=10)
    for row in range(2): axes[row, col].axis('off')

axes[0,0].set_ylabel('元画像', fontsize=12, rotation=0, labelpad=40)
axes[1,0].set_ylabel('再構成
(CFG)', fontsize=12, rotation=0, labelpad=40)

fig.suptitle('CFGスケール w が Inversion→再構成 に与える累積誤差', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_314_01_cfg_reconstruction_error.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 w=1（CFGなし）は完璧に再構成できるが、wが上がると誤差が蓄積して崩れる")

---

## 2. Null-text Inversion

### なぜ w=1 (CFGなし) で Inversion して、Sampling を w>1 ですればいいのでは？
→ それをすると**出力が元画像から大きくランダムに乖離**してしまいます（構造が保てない）。

### Null-text 最適化のアイデア

1. **w=1 で Inversion**（完璧な再構成軌跡 $ x_{T}, x_{T-1}, ... $ を記録）
2. **Sampling は w>1（目標のCFG強度）で実行**
3. ただし、各ステップ $t$ で、軌跡から逸脱しないように **Null-textベクトル $ \varnothing_t $ を最適化**する

$$\min_{\varnothing_t} || x_{t-1}^* - \text{DDIM}(x_t, \varnothing_t, c, w) ||^2$$
（目標は「w=1の軌跡」に「w>1のサンプリング」を一致させる無条件ベクトルを見つけること）

In [ ]:
# ============================================================
# 簡易版 Null-text Inversion: トラジェクトリー最適化
# ============================================================

def null_text_optimization(model, x_0, c_label, w_target=7.5, n_steps=20, opt_iters=10):
    """各タイムステップの無条件ベクトルを最適化する"""
    ab = schedule['alpha_bars'].to(device)
    step_idx = np.linspace(0, 99, n_steps + 1, dtype=int)

    # 1. Pivot Trajectoryの設定 (w=1でのInversion)
    x_T, pivot_traj = ddim_inversion_cfg(model, x_0, c_label, w=1.0, n_steps=n_steps)
    pivot_traj.reverse() # [x_T, x_{T-1}, ..., x_0]

    # 無条件ベクトルのリスト（初期値はデフォルトNull）
    null_embs = [model.null_token.unsqueeze(0).clone().detach() for _ in range(n_steps)]

    x = x_T.clone()
    c = torch.full((x.shape[0],), c_label, dtype=torch.long).to(device)

    reconstructed_traj = []

    for i in range(n_steps, 0, -1):
        t_c, t_p = step_idx[i], step_idx[i - 1]
        t_t = torch.full((x.shape[0],), t_c, dtype=torch.long).to(device)

        # ターゲット（w=1の軌跡における期待される一つ先の状態）
        target_x_prev = pivot_traj[n_steps - i + 1].clone().detach()

        # このステップのNull埋め込みを最適化パラメータに
        uncond_emb = null_embs[n_steps - i].clone().requires_grad_(True)
        optimizer = optim.Adam([uncond_emb], lr=0.01)

        for _ in range(opt_iters):
            optimizer.zero_grad()
            # カスタム無条件ベクトルを使ったCFG
            eps = model.forward_cfg(x.detach(), t_t, c, w=w_target, custom_null=uncond_emb)
            x0_p = (x.detach() - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
            pred_x_prev = torch.sqrt(ab[t_p]) * x0_p + torch.sqrt(1 - ab[t_p]) * eps

            # 軌跡の誤差を最小化
            loss = nn.functional.mse_loss(pred_x_prev, target_x_prev)
            loss.backward()
            optimizer.step()

        # 最適化後のベクトルで実際のステップを進める
        with torch.no_grad():
            eps = model.forward_cfg(x, t_t, c, w=w_target, custom_null=uncond_emb)
            x0_p = (x - torch.sqrt(1 - ab[t_c]) * eps) / torch.sqrt(ab[t_c])
            x = torch.sqrt(ab[t_p]) * x0_p + torch.sqrt(1 - ab[t_p]) * eps

        null_embs[n_steps - i] = uncond_emb.detach()

    return x, null_embs

# テスト: w=7.5の条件下で、Null-text Inversionで再構成を試みる
# （注: 非常に簡易化された実装なのでMNISTでは効果が限定的ですが、概念は伝わります）
print("Null-text最適化によるサンプリング中...")
w_high = 7.5
x_recon_null, optimized_nulls = null_text_optimization(cond_model, img_5, 5, w_target=w_high, n_steps=20, opt_iters=30)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(((img_5[0].cpu().numpy().reshape(28,28)+1)/2).clip(0,1), cmap='gray'); axes[0].set_title('元画像', fontsize=12)
x_T_w1, _ = ddim_inversion_cfg(cond_model, img_5, 5, w=1.0)
x_recon_bad = ddim_sample_cfg(cond_model, x_T_w1, 5, w=w_high)
axes[1].imshow(((x_recon_bad[0].cpu().numpy().reshape(28,28)+1)/2).clip(0,1), cmap='gray'); axes[1].set_title(f'Inv(w=1) → Smp(w={w_high})
構造が崩れる', fontsize=11)
x_T_wH, _ = ddim_inversion_cfg(cond_model, img_5, 5, w=w_high)
x_recon_bad2 = ddim_sample_cfg(cond_model, x_T_wH, 5, w=w_high)
axes[2].imshow(((x_recon_bad2[0].cpu().numpy().reshape(28,28)+1)/2).clip(0,1), cmap='gray'); axes[2].set_title(f'Inv(w={w_high}) → Smp(w={w_high})
累積誤差で崩れる', fontsize=11)
axes[3].imshow(((x_recon_null[0].cpu().numpy().reshape(28,28)+1)/2).clip(0,1), cmap='gray'); axes[3].set_title(f'Null-text Inv → Smp(w={w_high})
構造を保持！', fontsize=11, fontweight='bold', color='red')

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('fig_314_02_null_text_result.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 3. Prompt-to-Prompt の直感

Null-text Inversionは**実画像の構造を保持**するための技術でした。
構造を保ったまま「猫」を「犬」に変えるにはどうすればいいでしょうか？

これを実現するのが **Prompt-to-Prompt (P2P)** 論文の「Cross-Attention入れ替え」です。

### 最も重要な知見

> **画像の空間的構造（レイアウトや形状）は、U-Net内部の Cross-Attention マップによって決定されている。**

- **元プロンプト**: "A picture of a **cat**"
- **新プロンプト**: "A picture of a **dog**"

P2Pでは、新プロンプトで画像を生成しながら、**U-NetのAttentionマップだけを元プロンプトの生成過程からコピー（注入）**します。

結果: 「猫の構造（耳の位置、姿勢）」を持った「犬」が生成されます。
*（※実装は数十層のAttention層をHooksで改変する必要があり複雑なため、概念のみに留めます。）*

---

## まとめ

### 🎯 学んだこと

- ✓ CFG下ではDDIM Inversionの「往復で元に戻る」仮定が崩れ、累積誤差が生じる
- ✓ Null-text Inversionは無条件埋め込み$ \varnothing $を微調整することで誤差を吸収し、高精度な再構成とCFG編集を両立する
- ✓ P2PはU-NetのAttention空間を操作し、画像の「構造」と「意味（プロンプト）」を分離して編集する

### ✅ 学習チェックリスト

- [ ] なぜw=7.5でInversionすると画像が崩れるのか説明できるか？
- [ ] P2Pがどこを操作して構造を保持しているか説明できるか？

---

**次のステップ**: ノートブック315「Pix2PixとControlNet」で、画像条件（エッジや骨格）による制御への応用を学びます！